In [ ]:
import os

os.environ["OPENAI_API_KEY"] = "your-api-key-here"

from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

loader = PyPDFLoader("example_document.pdf")
docs = loader.load()

print(f"Loaded {len(docs)} pages")

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = FAISS.from_documents(docs, embeddings)

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

prompt = ChatPromptTemplate.from_template("""
Answer the question using only the provided document context.

Context:
{context}

Question:
{question}
""")

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

chain = (
    {
        "context": retriever,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

question = "What are the main rules and obligations described in this document?"

answer = chain.invoke(question)

print(answer)